# More on cleaning data in Pandas

In [55]:
x = 0
try:
    # If x is not 10, raise an AssertionError with the message
    assert type(x)==float, "x should be a float number" 
    print("x is 10. Program continues.")
except AssertionError as e:
    print(f'error is {e}')

error is x should be a float number


In [56]:
import os
import pandas as pd
from pathlib import Path

file_name=Path.cwd()
ride_sharing=pd.read_csv(file_name/'data'/'ride_sharing_new.csv')

ride_sharing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25760 entries, 0 to 25759
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Unnamed: 0       25760 non-null  int64 
 1   duration         25760 non-null  object
 2   station_A_id     25760 non-null  int64 
 3   station_A_name   25760 non-null  object
 4   station_B_id     25760 non-null  int64 
 5   station_B_name   25760 non-null  object
 6   bike_id          25760 non-null  int64 
 7   user_type        25760 non-null  int64 
 8   user_birth_year  25760 non-null  int64 
 9   user_gender      25760 non-null  object
dtypes: int64(6), object(4)
memory usage: 2.0+ MB


Showing summary stats of variables in the df

In [57]:
ride_sharing['user_type'].describe()

count    25760.000000
mean         2.008385
std          0.704541
min          1.000000
25%          2.000000
50%          2.000000
75%          3.000000
max          3.000000
Name: user_type, dtype: float64

convert date type to category and verify

In [58]:
ride_sharing['user_type']=ride_sharing['user_type'].astype('category')

assert ride_sharing['user_type'].dtype=='category'

```string.strip(<expression>)``` is stripping certain string

```string.replace(old, new, count)``` is also common

```string.len()``` is also common

In [ ]:
ride_sharing['duration_trim'] = ride_sharing['duration'].str.strip('minutes')
ride_sharing['duration_time'] = ride_sharing['duration_trim'].astype('int')

print(ride_sharing['duration_time'].head(2), ride_sharing['duration_trim'].head(2))



0    12
1    24
Name: duration_time, dtype: int64 0    12 
1    24 
Name: duration_trim, dtype: object


```
DataFrame.drop(labels=None, axis=0, index=None, columns=None, level=None, inplace=False, errors='raise')
```

* labels: Specifies the index or column labels to drop (single label or list-like).
* axis: Determines whether to drop labels from the index (rows, 0 or 'index') or columns (1 or 'columns'), with a default of 0.
* index: An alternative to specifying axis=0 for labels to drop from the index.
* columns: An alternative to specifying axis=1 for labels to drop from columns.
* inplace: A boolean parameter (default False). If True, the operation modifies the original DataFrame and returns None.
* errors: Controls error handling for labels not found (default 'raise'). If set to 'ignore', errors are suppressed. 

```pd.to_datetime()``` converts to timestampt. 

Build-in ```Datetime``` requires three parameters so it works the best with ```date``` format. 


In [60]:
print(ride_sharing['user_birth_year'].describe())

count    25760.000000
mean      1983.054969
std         10.010992
min       1901.000000
25%       1978.000000
50%       1985.000000
75%       1990.000000
max       2001.000000
Name: user_birth_year, dtype: float64


In [61]:
import datetime as dt
from datetime import timedelta


ride_sharing['user_dt'] = pd.to_datetime(ride_sharing['user_birth_year']).dt.date

# Save today's date
today = dt.date.today()-timedelta(days=360)

# Set all in the future to today's date
ride_sharing.loc[ride_sharing['user_dt'] > today, 'user_dt'] = today
ride_sharing['user_dt'].head(2)


0    1970-01-01
1    1970-01-01
Name: user_dt, dtype: object

3 ways of selection

|  | `df[]` | `.loc` (Label) | `.iloc` (Position) |
| :--- | :--- | :--- | :--- |
| **Select Columns** | `df[['A', 'B']]` | `df.loc[:, ['A', 'B']]` | `df.iloc[:, [0, 1]]` |
| **Select Rows** | `df[0:3]` | `df.loc['row_a':'row_c']` | `df.iloc[0:3]` |
| **Rows & Columns** | N/A | `df.loc['row_a', 'Col_A']` | `df.iloc[0, 0]` |


duplicate values
```df.duplicated(subset=None, keep='first')```
* subset: (Optional) return a Boolean Series.
* keep: Controls which duplicates (if any) are marked:
    * 'first' (default): Marks all duplicates as True except for the first occurrence.
    * 'last': Marks all duplicates as True except for the last occurrence.
    * False: Marks all occurrences of duplicate rows as True. 

remove duplicates
```drop_duplicates(subset=None, keep='first', inplace=False, ignore_index=False)```
*   `subset`: List of column names to check for uniqueness (e.g., `subset=['ID']`).
*   `keep`: 
    *   `'first'` (default): Keep the first appearance.
    *   `'last'`: Keep the last appearance.
    *   `False`: Drop **all** occurrences of the duplicate.
*   `inplace`: If `True`, modifies the existing DataFrame instead of returning a new one.

remove rows or columns that contain missing values (`NaN`, `None`)
```df.dropna()```
*   `subset`: List of columns to look in. Only rows with missing values in *these* specific columns will be dropped.
*   `how`:
    *   `'any'` (default): Drop row if **any** value in the subset is missing.
    *   `'all'`: Drop row only if **all** values in the subset are missing.
*   `thresh`: Require a minimum number of non-NA values to keep the row (e.g., `thresh=3` means keep if at least 3 values areNOT null).
*   `axis`: 
    *   `0` or `'index'` (default): Drops rows.
    *   `1` or `'columns'`: Drops entire columns containing nulls.

```duplicates``` differs from ```drop_duplicates``` by returning a series of True or False to be further passed to subsetting like in 
```
all_dupes = df[df.duplicated(subset=['ID'], keep=False)]
summary_stats = all_dupes.groupby('ID')['Sales'].agg(['sum', 'mean', 'count'])
```.



more on dropping data: checking unique values



In [62]:
airlines=pd.read_csv(file_name/'data'/'airlines_final.csv')
airlines.head(2)

,Unnamed: 0,id,day,airline,destination,dest_region,dest_size,boarding_area,dept_time,wait_min,cleanliness,safety,satisfaction
0,0,1351,Tuesday,UNITED INTL,KANSAI,Asia,Hub,Gates 91-102,2018-12-31,115.0,Clean,Neutral,Very satisfied
1,1,373,Friday,ALASKA,SAN JOSE DEL CABO,Canada/Mexico,Small,Gates 50-59,2018-12-31,135.0,Clean,Very safe,Very satisfied


In [63]:
# unique values
airlines['safety'].unique()

array(['Neutral', 'Very safe', 'Somewhat safe', 'Very unsafe',
       'Somewhat unsafe'], dtype=object)

In [64]:
# find what is in airlines['safety'] but not in ['Neutral', 'Very safe']
# i.e. anti join airlines to ['Neutral', 'Very safe']
cat_clean = set(airlines['safety']).difference(['Neutral', 'Somewhat safe'])

# filter columns in ['Neutral', 'Very safe']
airlines_new=airlines[airlines['safety'].isin(['Neutral', 'Somewhat safe'])]
airlines_new.head(2)

# filter columns not in ['Neutral', 'Very safe']
airlines_new2=airlines[~airlines['safety'].isin(['Neutral', 'Somewhat safe'])]
airlines_new2.head(2)


,Unnamed: 0,id,day,airline,destination,dest_region,dest_size,boarding_area,dept_time,wait_min,cleanliness,safety,satisfaction
1,1,373,Friday,ALASKA,SAN JOSE DEL CABO,Canada/Mexico,Small,Gates 50-59,2018-12-31,135.0,Clean,Very safe,Very satisfied
3,3,1157,Tuesday,SOUTHWEST,LOS ANGELES,West US,Hub,Gates 20-39,2018-12-31,190.0,Clean,Very safe,Somewhat satsified



| Feature | `pd.cut` (What you have) | `pd.qcut` (Quantiles) |
| :--- | :--- | :--- |
| **Input List** | `[0, 60, 180, np.inf]` | `[0, 0.25, 0.75, 1.0]` |
| **Logic** | "Put everyone waiting **under 60 mins** in 'short'." | "Put the **quickest 25% of people** in 'short'." |
| **Outcome** | 'short' could have 10 people or 1,000 people. | 'short' will always have **exactly 25%** of your rows. |
| **Wait Times** | $(0, 60]$,  $(60, 180]$... | $(0, 0.25]$,  $(0.25, 0.75]$... |


In [69]:
import numpy as np
# Create ranges for categories
label_ranges = [0, 60, 180 , np.inf]
label_names = ['short', 'medium', 'long']

# Create wait_type column
airlines['wait_type'] = pd.cut(airlines['wait_min'], bins = label_ranges, 
                                labels = label_names)


# Create wait_type column
label_qranges = [0, 0.25 ,0.75, 1]
airlines['wait_type2'] = pd.qcut(airlines['wait_min'], q = label_qranges, 
                                labels = label_names)


changing the existing label of categories

| Feature | `.cat.rename_categories()` | `.replace()` |
| :--- | :--- | :--- |
| **Data Type** | Stays `category` | Often converts to `object` |
| **Safety** | High (prevents invalid labels) | Low (can introduce any value) |
| **Efficiency** | High (modifies metadata) | Lower (scans every row) |


In [70]:
# Create mappings and replace
mappings = {'Monday':'weekday', 'Tuesday':'weekday', 'Wednesday': 'weekday', 
            'Thursday': 'weekday', 'Friday': 'weekday', 
            'Saturday': 'weekend', 'Sunday': 'weekend'}

airlines['day_week'] = airlines['day'].replace(mappings)

#### Handling missing values
* ```df.loc[<expression>, <expression>] = np.nan``` convert to NaN
* ```df[[<column>]].isna()``` checks missing values
* ```df.fillna({column_name: values_to_be_filled})``` to replace NA
* ```df.dropna(subset=[column_name])``` to drop rows with NA


### case_when

In [2]:
import pandas as pd
import numpy as np

# -------------------------
# Example data (like tidyverse tibble)
# -------------------------
df = pd.DataFrame({
    "country": ["CA", "CA", "SA", "SA", "MX", "CA"],
    "x":       [  5,   12,   10,   25,    7,    0],
})

# ---------------------------------------------------
# "case_when" (pythonic) template using np.select
# ---------------------------------------------------
condlist=[
        (df["country"].eq("CA") & df["x"].le(10)),   # case 1
        (df["country"].eq("SA") & df["x"].le(20)),   # case 2
        # add more cases here...
    ]

choicelist=[
        2 * df["x"],   # result for case 1
        3 * df["x"],   # result for case 2
        # corresponding results...
    ]

df["y"] = np.select(condlist, choicelist,default=df["x"])

print(df.head())

  country   x   y
0      CA   5  10
1      CA  12  12
2      SA  10  30
3      SA  25  25
4      MX   7   7


# Date time

## conversion

- Convert string to datetime: `pd.to_datetime(df['timestamp'])`
- Extract date only (removes time portion): `df['timestamp'].dt.date`
- Extract individual components using `.dt` accessor
    - `df['timestamp'].dt.year`
    - `df['timestamp'].dt.month`
    - `df['timestamp'].dt.day`
    - `df['timestamp'].dt.hour`
    - `df['timestamp'].dt.minute`
    - `df['timestamp'].dt.day_name()  `
    - `df['timestamp'].dt.quarter`

In [12]:
import pandas as pd

# Sample data
df = pd.DataFrame({
    'timestamp': ['2024-01-15 14:30:25', '2024-02-20 09:15:30', '2024-03-10 18:45:12']
})

df['timestamp']=pd.to_datetime(df['timestamp'])
print(df['timestamp'].dt.date)


0    2024-01-15
1    2024-02-20
2    2024-03-10
Name: timestamp, dtype: object


## rolling window

- two usages of `.rolling()`
    - `df['sales'].rolling(window=7).apply(lambda x: x.max() - x.min())`
    - `df['sales'].rolling(window=7,center=True).mean()`
- time delta
    - `df['date'] - df['date'].iloc[0]).dt.days`
    - using `.diff()`: `df['date'].diff().dt.days`
- changes
    - using `.diff()`: `df['sales'].diff(periods=2)`
    - downward shifting: `df['p']-df['p'].shift(1)`
- for filling NA
    - `.fillna(method='ffill')`

Key Functions Summary

| Function | Purpose | Example |
|----------|---------|---------|
| `.diff()` | Absolute difference from previous row | `df['price'].diff()` |
| `.diff(n)` | Difference from n rows back | `df['price'].diff(3)` |
| `.pct_change()` | Percentage change from previous | `df['price'].pct_change()` |
| `.shift(n)` | Move values up/down by n rows | `df['price'].shift(1)` |
| `.cumsum()` | Cumulative sum | `df['change'].cumsum()` |
| `.cumprod()` | Cumulative product | `(1 + returns).cumprod()` |
| `dt.days` | Extract days from timedelta | `(date2 - date1).dt.days` |
| `dt.total_seconds()` | Total seconds in timedelta | `(date2 - date1).dt.total_seconds()` |
